# Geneformer Embedding Pipeline
**Steps:**
1. Environment setup (clone repo, install deps)
2. Load raw AnnData cohort from Zarr
3. Map HUGO gene symbols → Ensembl IDs
4. Prepare pseudo-counts + tokenize for Geneformer
5. Extract embeddings (Geneformer V2 104M cancer model)
6. Compute UMAP and save figures

## 0. Environment Setup

In [ ]:
import subprocess
import os

subprocess.run(["pip", "install", "-e", "."], check=True)
subprocess.run(["pip", "install", "-q", "peft==0.10.0"], check=True)
subprocess.run(["pip", "install", "-q", "loguru", "mygene"], check=True)
subprocess.run(["pip", "install", "-q", "transformers<4.40.0"], check=True)
subprocess.run(["pip", "install", "-q", "anndata==0.10.9", "scanpy==1.10.1"], check=True)
subprocess.run(["pip", "install", "-q", "git+https://huggingface.co/ctheodoris/Geneformer.git"], check=True)
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "clone", "https://huggingface.co/ctheodoris/Geneformer", "/content/Geneformer"], check=True)

from google.colab import drive
drive.mount("/content/drive")

## 1. Imports

In [ ]:
from loguru import logger

from batchcor_rna_emb.config import (
    BATCH_COL,
    DATASET_PATH,
    DIAGNOSIS_COL,
    FIGURES_DIR,
)
from batchcor_rna_emb.data_io import load_cohort
from batchcor_rna_emb.feature_calc.gene_mapping_geneformer import (
    prepare_pseudo_counts,
    rename_adata_vars_to_ensembl,
    tokenize_adata,
)
from batchcor_rna_emb.geneformer_embedder import build_embedding_adata, extract_embeddings
from batchcor_rna_emb.visualization.plots import (
    plot_umap_grid,
    plot_metrics_heatmap,
    plot_scatter_avg_batch_bio,
    plot_roc_curves,
    plot_generalization_decay,
    plot_loss_curves,
)

## 2. Load Raw Data

In [ ]:
logger.info("Loading cohort from: {}", DATASET_PATH)
adata = load_cohort(DATASET_PATH)

logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)
logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
logger.info("Diagnosis distribution:\n{}", adata.obs[DIAGNOSIS_COL].value_counts())

## 3. Map HUGO Gene Symbols → Ensembl IDs

In [ ]:
adata_mapped, found_genes, missing_genes = rename_adata_vars_to_ensembl(adata)
logger.info(
    "After mapping: {} genes retained, {} dropped",
    len(found_genes), len(missing_genes),
)

## 4. Prepare Pseudo-counts & Tokenize

In [ ]:
adata_mapped = prepare_pseudo_counts(adata_mapped)
tokenize_adata(adata_mapped)

## 5. Extract Geneformer Embeddings

In [ ]:
embeddings, metadata = extract_embeddings()

adata_emb = build_embedding_adata(
    embeddings=embeddings,
    metadata=metadata,
    adata_mapped=adata_mapped,
    extra_cols=["OS", "PFS", "Response", "Therapy"],
)

logger.info("Embedding AnnData: {}", adata_emb.shape)
logger.info(
    "Diagnosis value counts:\n{}",
    adata_emb.obs["diagnosis"].value_counts(dropna=False),
)
logger.info(
    "Batch value counts:\n{}",
    adata_emb.obs["RNA_batch"].value_counts(dropna=False),
)

## 6. UMAP + Visualizations

Using `plot_umap_grid` from `batchcor_rna_emb.visualization.plots` to visualise
the embedding coloured by **diagnosis** and **RNA batch**.

In [ ]:
import scanpy as sc

# Compute neighbors + UMAP on the embedding AnnData
sc.pp.neighbors(adata_emb, use_rep="X", n_neighbors=15, metric="cosine")
sc.tl.umap(adata_emb, min_dist=0.3)

logger.info("UMAP computed. Shape of X_umap: {}", adata_emb.obsm["X_umap"].shape)

In [ ]:
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig = plot_umap_grid(
    adatas={"Geneformer V2": adata_emb},
    color_cols=["diagnosis", "RNA_batch"],
    basis_key="X_umap",
    figsize=(6, 5),
    point_size=8,
    save_path=str(FIGURES_DIR / "umap_diagnosis_batch.png"),
)
fig.show()

### 6b. Additional plots (run after batch correction results are available)

The cells below show how to use the other functions from `visualization/plots.py`.
They are left with placeholder data — fill in real results as the project progresses.

In [ ]:
# plot_metrics_heatmap — run after batch correction benchmarking
# import pandas as pd
# metrics_df = pd.read_csv(FIGURES_DIR / "metrics_table.csv", index_col=0)
# fig_heatmap = plot_metrics_heatmap(
#     df=metrics_df,
#     title="Batch Correction Metrics",
#     save_path=str(FIGURES_DIR / "metrics_heatmap.png"),
# )
# fig_heatmap.show()

In [ ]:
# plot_scatter_avg_batch_bio — batch vs bio preservation trade-off
# fig_scatter = plot_scatter_avg_batch_bio(
#     df=metrics_df,
#     save_path=str(FIGURES_DIR / "batch_bio_scatter.png"),
# )
# fig_scatter.show()

In [ ]:
# plot_roc_curves — ICI response prediction ROC
# import numpy as np
# roc_results = {
#     "Geneformer V2": (y_true, y_prob),   # fill in real arrays
#     "Harmony": (y_true_h, y_prob_h),
# }
# fig_roc = plot_roc_curves(
#     results=roc_results,
#     title="ICI Response Prediction ROC",
#     save_path=str(FIGURES_DIR / "roc_curves.png"),
# )
# fig_roc.show()

In [ ]:
# plot_generalization_decay — stress test results
# import pandas as pd
# stress_df = pd.read_csv(FIGURES_DIR / "stress_results.csv")
# fig_decay = plot_generalization_decay(
#     results_df=stress_df,
#     metric_col="f1_weighted",
#     level_col="level",
#     group_col="model",
#     save_path=str(FIGURES_DIR / "generalization_decay.png"),
# )
# fig_decay.show()

In [ ]:
# plot_loss_curves — DANN training loss
# fig_loss = plot_loss_curves(
#     history=dann_trainer.history,   # fill in real DANN history dict
#     title="DANN Training Loss",
#     save_path=str(FIGURES_DIR / "dann_loss.png"),
# )
# fig_loss.show()